In [ ]:
import pandas as pd

# File paths
all_path = r"D:\DATA\full_dataset\with_snomed_category.csv"

# Load datasets
df_all = pd.read_csv(all_path)

In [ ]:
from helper_functions import strings2lists

list_str_cols = ['snomed_code', 'M', 'T', 'snomed_text', 'T_text', 'M_text', 'undersoeger_anonymous', 'T_category', 'M_category']

for col in list_str_cols: 
    df_all[col] = df_all[col].apply(strings2lists)

In [ ]:
# Add class idx for training
class_dict = {
    'Normal Tissue': 0, 
    'Morphology Not Applicable / Insufficient Tissue': 0, 
    'Cellular Changes / Abnormal Tissue Structure': 1,
    'Traumatic Lesions': 1, 
    'Congenital Malformations': 1,
    'Pregnancy-Related Tissues/Changes': 1,
    'Obstruction / Fluid Retention / Cysts': 1, 
    'Mechanical Changes / Architectural Distortion': 1,
    'Inflammation': 1, 
    'Fibrosis': 1, 
    'Degeneration / Necrosis / Atrophy': 1, 
    'Material Deposits': 1, 
    'Resection Margin Free': 0, 
    'Resection Margin Uncertain': 1,
    'Resection Margin Not Free': 1, 
    'Proliferative/Pre-neoplastic Changes': 1, 
    'Benign Neoplasm': 1, 
    'Uncertain / Borderline Neoplasm': 1, 
    'In Situ Neoplasm': 1, 
    'Malignant Neoplasm': 1,
}

df_all["M_idx"] = df_all["M_category"].apply(lambda lst: [class_dict[x] for x in lst])
df_all['M_idx'] = df_all['M_idx'].apply(lambda x: max(x) if isinstance(x, list) else x)

# 0: Healthy
# 1: Non-healthy

In [ ]:
from helper_functions import subset_df, subset_df_list, subset_df_processed

# Filter: HE stain
df_HE = subset_df(df_all, "stain", "HE")

# Filter: tissue detection complete
cache_file = r"D:\DATA\cache_tissue_artifact_features.pkl"
df_error = subset_df_processed(df_HE, cache_file, category="tissue", status="error", model="default")
paths_error = df_error["filename"].tolist()

df_tissue = subset_df_processed(df_HE, cache_file, category="tissue", status="complete", model="default")
paths_tissue = df_tissue["filename"].tolist()


df_HE = df_HE[~df_HE["filename"].isin(paths_error)]
print(len(df_HE))
df_HE = df_HE[df_HE["filename"].isin(paths_tissue)]
df_HE["T_category"].value_counts()

In [ ]:
from collections import Counter
import numpy as np
import pandas as pd

def _flatten_list(x):
    flat = []
    if isinstance(x, list):
        for item in x:
            if isinstance(item, list):
                flat.extend(item)
            else:
                flat.append(item)
    else:
        flat.append(x)
    return flat

def count_tissue_frequencies(df, tissue_col="T_category"):
    counter = Counter()
    for x in df[tissue_col]:
        flat = _flatten_list(x)
        counter.update(flat)
    return counter

def contains_any_tissues(x, tissues):
    flat = _flatten_list(x)
    return any(t in flat for t in tissues)

def pick_rep_tissue_by_global_freq(x, tissue_counts):
    """
    For a slide's tissue list, return the tissue with the highest global count.
    If tie, returns the first among ties (arbitrary but deterministic).
    """
    flat = _flatten_list(x)
    if not flat:
        return None
    return max(flat, key=lambda t: tissue_counts.get(t, 0))

In [ ]:
# Top n
top_n = 5

# Count tissue frequencies (multi-label aware)
tissue_counts = count_tissue_frequencies(df_HE, "T_category")

# Derive top n tissues automatically
top_tissues = [t for t, _ in tissue_counts.most_common(top_n)]

print(f"Top {top_n} tissues (by global frequency, multi-label aware):")
for t, c in tissue_counts.most_common(top_n):
    print(f"  {t}: {c}")

In [ ]:
# Assign each slide to the tissue with highest global frequency
df_HE["_tissue_rep"] = df_HE["T_category"].apply(lambda x: pick_rep_tissue_by_global_freq(x, tissue_counts))

# Filter to slides whose representative tissue is in the top 10
mask_top = df_HE["_tissue_rep"].isin(top_tissues)
df_top = df_HE[mask_top].reset_index(drop=True)

print(f"\nSlides per representative tissue (top {top_n}):")
print(df_top["_tissue_rep"].value_counts())

In [ ]:
print("\nSlides per tissue and M_idx:")
print(df_top.groupby(["_tissue_rep"])["M_idx"].value_counts())

In [ ]:
# Exclude slides with less than 20 in any idx
min_count = 20
expected_midx = [0, 1]

# Count rows for each T_category/M_idx combination
counts = df_top.groupby(["_tissue_rep", "M_idx"], observed=True).size().unstack(fill_value=0).reindex(columns=expected_midx, fill_value=0)

# Identify tissues where at least one M_idx count is below the threshold
tissues_to_remove = counts.index[(counts < min_count).any(axis=1)]

# Remove those entire tissues
df_top = df_top[
    ~df_top["_tissue_rep"].isin(tissues_to_remove)
].copy()

print("Removed tissues:")
print(tissues_to_remove.tolist())

In [ ]:
import numpy as np
import pandas as pd
from collections import defaultdict


def strict_patient_safe_split(
    df,
    tissue_col="_tissue_rep",
    patient_col="rekvnr",
    midx_col="M_idx",
    test_frac=0.2,
    random_state=42,
    min_test_patients_per_group=1,
):
    """
    Paper-ready patient-safe split for a cross-organ dataset.

    Key properties:
    - Patients are assigned GLOBALLY, so no patient can appear in both train and test.
    - All selected tissues are kept.
    - (tissue, M_idx) representation is encouraged in both splits where possible.
    - Small groups are handled conservatively.

    Returns:
        train_df, test_df, assignment_df, group_summary_df
    """
    rng = np.random.default_rng(random_state)
    df = df.copy().reset_index(drop=True)

    # Drop rows missing critical identifiers.
    df = df.dropna(subset=[patient_col, tissue_col, midx_col]).copy()

    # Build patient -> set of (tissue, M_idx) memberships.
    patient_groups = (
        df.groupby(patient_col)
          .apply(lambda x: set(zip(x[tissue_col], x[midx_col])))
          .to_dict()
    )

    # Count unique patients per group.
    group_to_patients = defaultdict(set)
    for patient, groups in patient_groups.items():
        for group in groups:
            group_to_patients[group].add(patient)

    group_counts = {
        group: len(patients)
        for group, patients in group_to_patients.items()
    }

    # Target number of test patients per group.
    target_test = {}
    for group, n_patients in group_counts.items():
        if n_patients <= 1:
            target_test[group] = 0
        else:
            target_test[group] = min(
                max(min_test_patients_per_group, int(round(n_patients * test_frac))),
                n_patients - 1,
            )

    # Process rare groups first because they are hardest to satisfy.
    groups_sorted = sorted(group_counts.keys(), key=lambda g: group_counts[g])

    patient_assignment = {}  # patient -> "train" or "test"
    test_group_counts = defaultdict(int)

    # Greedy global assignment.
    for group in groups_sorted:
        patients = list(group_to_patients[group])
        rng.shuffle(patients)

        needed = target_test[group] - test_group_counts[group]
        if needed <= 0:
            continue

        # Prefer currently unassigned patients for the test set.
        unassigned = [p for p in patients if p not in patient_assignment]
        take = unassigned[:needed]

        for p in take:
            patient_assignment[p] = "test"
            for g in patient_groups[p]:
                test_group_counts[g] += 1

    # Assign all remaining patients to train.
    for patient in patient_groups:
        if patient not in patient_assignment:
            patient_assignment[patient] = "train"

    assignment_df = pd.DataFrame({
        patient_col: list(patient_assignment.keys()),
        "split": list(patient_assignment.values()),
    })

    df2 = df.merge(assignment_df, on=patient_col, how="left")
    train_df = df2[df2["split"] == "train"].drop(columns=["split"]).reset_index(drop=True)
    test_df = df2[df2["split"] == "test"].drop(columns=["split"]).reset_index(drop=True)

    # Final safety check.
    overlap = set(train_df[patient_col].unique()) & set(test_df[patient_col].unique())
    if overlap:
        raise ValueError(f"Patient leakage detected: {len(overlap)} overlapping patients")

    # Summarize achieved group representation.
    def _summarize(split_df, split_name):
        out = (
            split_df.groupby([tissue_col, midx_col])
                   .agg(
                       n_slides=(patient_col, "size"),
                       n_patients=(patient_col, pd.Series.nunique),
                   )
                   .reset_index()
        )
        out["split"] = split_name
        return out

    train_summary = _summarize(train_df, "train")
    test_summary = _summarize(test_df, "test")
    group_summary_df = pd.concat([train_summary, test_summary], ignore_index=True)

    return train_df, test_df, assignment_df, group_summary_df


def sample_slides_patient_diverse(
    df,
    tissue_col="_tissue_rep",
    patient_col="rekvnr",
    midx_col="M_idx",
    max_per_group=300,
    random_state=42,
):
    """
    Sample up to max_per_group slides per (tissue, M_idx), prioritizing
    patient diversity.

    Rules:
    - If <= max_per_group slides exist, keep all.
    - Otherwise, take one slide per patient first.
    - If still below target, fill from remaining slides at random.
    """
    rng = np.random.default_rng(random_state)
    selected_parts = []

    for (tissue, midx), group_df in df.groupby([tissue_col, midx_col], dropna=False):
        group_df = group_df.copy()

        if len(group_df) <= max_per_group:
            selected_parts.append(group_df)
            continue

        patient_ids = group_df[patient_col].dropna().unique().tolist()
        rng.shuffle(patient_ids)

        chosen_idx = []

        # First pass: maximize number of unique patients.
        for pid in patient_ids:
            patient_rows = group_df[group_df[patient_col] == pid]
            row = patient_rows.sample(n=1, random_state=int(rng.integers(0, 1_000_000)))
            chosen_idx.extend(row.index.tolist())
            if len(chosen_idx) >= max_per_group:
                break

        # Second pass: if target not reached, allow extra slides from already represented patients.
        if len(chosen_idx) < max_per_group:
            remaining_idx = group_df.index.difference(chosen_idx)
            n_extra = min(max_per_group - len(chosen_idx), len(remaining_idx))
            if n_extra > 0:
                extra_idx = group_df.loc[remaining_idx].sample(
                    n=n_extra,
                    random_state=int(rng.integers(0, 1_000_000))
                ).index.tolist()
                chosen_idx.extend(extra_idx)

        selected_parts.append(group_df.loc[chosen_idx].copy())

    return pd.concat(selected_parts, ignore_index=True).reset_index(drop=True)


def summarize_groups(df, tissue_col="_tissue_rep", midx_col="M_idx", patient_col="rekvnr"):
    """Summarize slides and patients per (tissue, M_idx)."""
    return (
        df.groupby([tissue_col, midx_col], dropna=False)
          .agg(
              n_slides=(patient_col, "size"),
              n_patients=(patient_col, pd.Series.nunique),
          )
          .reset_index()
          .sort_values([tissue_col, midx_col])
          .reset_index(drop=True)
    )


def validate_split(train_df, test_df, tissue_col="_tissue_rep", midx_col="M_idx", patient_col="rekvnr"):
    """
    Validation helper for paper-ready cohort checking.
    Returns a dict of useful diagnostics.
    """
    train_patients = set(train_df[patient_col].dropna().unique())
    test_patients = set(test_df[patient_col].dropna().unique())
    overlap = train_patients & test_patients

    train_groups = set(zip(train_df[tissue_col], train_df[midx_col]))
    test_groups = set(zip(test_df[tissue_col], test_df[midx_col]))

    return {
        "n_train_slides": len(train_df),
        "n_test_slides": len(test_df),
        "n_train_patients": len(train_patients),
        "n_test_patients": len(test_patients),
        "n_overlap_patients": len(overlap),
        "groups_only_in_train": sorted(train_groups - test_groups),
        "groups_only_in_test": sorted(test_groups - train_groups),
    }


In [ ]:
train_df, test_df, assignment_df, split_summary = strict_patient_safe_split(
    df_top,
    tissue_col="_tissue_rep",
    patient_col="rekvnr",
    midx_col="M_idx",
    test_frac=0.2,
    random_state=42,
)

train_sel = sample_slides_patient_diverse(
    train_df,
    tissue_col="_tissue_rep",
    patient_col="rekvnr",
    midx_col="M_idx",
    max_per_group=20,
    random_state=42,
)

print("Train", summarize_groups(train_sel, "_tissue_rep", "M_idx", "rekvnr"))

In [ ]:
train_sel.to_csv(r"D:\DATA\abmil_audit_v2.csv",index=False)